# SmartCart | Customer Segmentation & Marketing Intelligence

**Goal:** discover interpretable customer groups from spending, purchasing channels, household, and engagement behavior.

**Workflow:** data quality → feature engineering → leakage-aware preprocessing → K selection → clustering → evaluation → segment profiles → business actions.

> This is an unsupervised segmentation project. Campaign response is excluded from model inputs and analyzed only after clusters are assigned.


## 1. Setup

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import silhouette_score
from src.segmentation import load_and_engineer, prepare_matrix, evaluate_k_values, fit_segmentation, make_cluster_profile

sns.set_theme(style="whitegrid", context="notebook")
DATA_PATH = Path("data/smartcart_customers.csv")


## 2. Load and validate data

In [ ]:
raw = pd.read_csv(DATA_PATH)
print(f"Rows: {raw.shape[0]:,} | Columns: {raw.shape[1]}")
display(raw.head())
quality = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "missing_count": raw.isna().sum(),
    "missing_pct": (raw.isna().mean() * 100).round(2),
    "unique_values": raw.nunique()
}).sort_values("missing_pct", ascending=False)
display(quality)


In [ ]:
customers = load_and_engineer(raw)
print(f"Rows retained after plausibility checks: {len(customers):,}")
display(customers[["Age", "Income", "Customer_Tenure_Days", "Total_Spending",
                   "Total_Purchases", "Total_Children", "Web_Purchase_Share"]].describe().T)


## 3. Explore customer behavior

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(customers["Total_Spending"], bins=35, ax=axes[0])
axes[0].set_title("Customer spending distribution")
sns.scatterplot(data=customers, x="Income", y="Total_Spending", alpha=.65, ax=axes[1])
axes[1].set_title("Income vs. total spending")
plt.tight_layout()


In [ ]:
channel_cols = ["NumWebPurchases", "NumCatalogPurchases", "NumStorePurchases"]
channel_summary = customers[channel_cols].mean().sort_values(ascending=False)
display(channel_summary.to_frame("average_purchases"))
channel_summary.plot(kind="bar", figsize=(8, 4), title="Average purchases by channel")
plt.ylabel("Average purchases")
plt.tight_layout()


## 4. Select the number of clusters

In [ ]:
evaluation = evaluate_k_values(customers, range(2, 9))
display(evaluation.round(4))
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.lineplot(data=evaluation, x="k", y="silhouette", marker="o", ax=axes[0])
axes[0].set_title("Silhouette (higher is better)")
sns.lineplot(data=evaluation, x="k", y="davies_bouldin", marker="o", ax=axes[1])
axes[1].set_title("Davies–Bouldin (lower is better)")
sns.lineplot(data=evaluation, x="k", y="inertia", marker="o", ax=axes[2])
axes[2].set_title("Inertia / elbow")
plt.tight_layout()


### Choosing K
Do not automatically pick the maximum silhouette or an elbow alone. Consider metric trade-offs, cluster sizes, stability across random seeds, and whether each segment suggests a distinct marketing action. The next cell uses K=3 as a starting point; change it after reviewing the evaluation table.

In [ ]:
K = 3
result = fit_segmentation(customers, k=K)
labeled = customers.copy()
labeled["Cluster"] = result["labels"]
print("Model metrics:", {k: round(v, 4) for k, v in result["metrics"].items()})


## 5. Visualize clusters (PCA is for plotting only)

In [ ]:
projection = pd.DataFrame(result["projection"], columns=["PC1", "PC2"])
projection["Cluster"] = labeled["Cluster"].astype(str).values
projection["Total_Spending"] = labeled["Total_Spending"].values
fig, ax = plt.subplots(figsize=(9, 6))
sns.scatterplot(data=projection, x="PC1", y="PC2", hue="Cluster", alpha=.75, s=45, ax=ax)
ax.set_title("Customer segments projected into 2D PCA space")
plt.tight_layout()


## 6. Interpret segments

In [ ]:
profile = make_cluster_profile(labeled, labeled["Cluster"])
display(profile)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.barplot(data=profile, x="Cluster", y="Total_Spending", ax=axes[0])
axes[0].set_title("Average spending by segment")
sns.barplot(data=profile, x="Cluster", y="Campaign_Response_Rate_%", ax=axes[1])
axes[1].set_title("Historical campaign response by segment")
axes[1].set_ylabel("Response rate (%)")
plt.tight_layout()


## 7. Translate findings into testable actions

In [ ]:
median_spend = profile["Total_Spending"].median()
median_recency = profile["Recency"].median()
actions = []
for _, row in profile.iterrows():
    if row["Total_Spending"] >= median_spend:
        action = "Test VIP benefits, premium bundles, and early access."
    elif row["Recency"] >= median_recency:
        action = "Test a win-back message with a relevant personalized offer."
    else:
        action = "Test complementary-product recommendations and a modest incentive."
    actions.append({"Cluster": int(row["Cluster"]), "Customers": int(row["Customers"]),
                    "Customer_Share_%": row["Customer_Share_%"], "Suggested_test": action})
action_plan = pd.DataFrame(actions)
display(action_plan)


## 8. Export artifacts
Exports are generated locally for further analysis or dashboard use.

In [ ]:
Path("outputs").mkdir(exist_ok=True)
profile.to_csv("outputs/smartcart_segment_profile.csv", index=False)
labeled.to_csv("outputs/smartcart_customers_with_segments.csv", index=False)
evaluation.to_csv("outputs/smartcart_cluster_evaluation.csv", index=False)
print("Saved outputs/segment profile, labeled customers, and evaluation table.")


## 9. Limitations and next steps
- Segment response rates are descriptive; they do not prove a campaign caused a response.
- Run stability checks across multiple seeds and consider a holdout campaign experiment before deploying offers.
- Add a business-defined value metric (margin or revenue) if available; spending amounts alone are not profit.
- If this dataset is used for a public portfolio, state clearly that it is a historical customer marketing dataset rather than live SmartCart transactions.
- Consider adding SQL analysis and a Power BI dashboard as a next phase.
